In [ ]:
# KALMAN Historical Quant 2017 -> Now / Colab v3
# Research / BACKTEST / SHADOW only.
# No Toss orders. No Neon writes. No LIVE execution.

from google.colab import drive

import json
import os
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

# ============================================================
# USER CONFIG
# ============================================================
START_DATE = "2017-01-01"
FORCE_REBUILD_MATRICES = False
RECREATE_VENVS = True
ENABLE_QLIB_RECORDER = False
RUN_RISKFOLIO = True

TRAIN_OBS = 504
VALID_OBS = 63
TEST_OBS = 126
MAX_HOLD_BARS = 20

PORTFOLIO_METHOD = "hrp"
PORTFOLIO_LOOKBACK = 180
PORTFOLIO_MIN_OBS = 90
PORTFOLIO_REBALANCE = "M"

MYDRIVE = Path("/content/drive/MyDrive")
REPO_ROOT = Path("/content/Codex_kalman_v3")
KALMAN_VENV = Path("/content/.venv-kalman-v3")
RISK_VENV = Path("/content/.venv-riskfolio-v3")
MIRROR_DIR = Path("/content/kalman_matrix_mirror_v3")

CURRENT_PHASE = "BOOTSTRAP"


def phase(name):
    global CURRENT_PHASE
    CURRENT_PHASE = str(name)
    print("\n" + "=" * 88)
    print(f"PHASE: {CURRENT_PHASE}")
    print("=" * 88)


def run(cmd, *, cwd=None, env=None, quiet=False):
    cmd = [str(x) for x in cmd]
    if not quiet:
        print("\n$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, env=env)


def capture(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd))
    return subprocess.check_output(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()


def create_venv(path):
    path = Path(path)
    if RECREATE_VENVS and path.exists():
        shutil.rmtree(path)

    if (path / "bin/python").exists():
        return

    try:
        run([sys.executable, "-m", "venv", str(path)])
    except subprocess.CalledProcessError:
        run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        run([sys.executable, "-m", "virtualenv", str(path)])

    if not (path / "bin/python").exists():
        raise RuntimeError(f"virtualenv creation failed: {path}")


def bounded_find_dir(root, predicate, max_depth=5):
    root = Path(root)
    if not root.exists():
        return None
    base_depth = len(root.parts)
    for current, dirs, _files in os.walk(root):
        current = Path(current)
        depth = len(current.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        if predicate(current):
            return current
    return None


def find_model_root():
    candidates = [
        MYDRIVE / "Market_Model_V2",
        MYDRIVE / "Kalman" / "Market_Model_V2",
        MYDRIVE / "kalman" / "Market_Model_V2",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    return bounded_find_dir(
        MYDRIVE,
        lambda p: p.name == "Market_Model_V2",
        max_depth=5,
    )


def find_market_root(model_root):
    candidates = [
        model_root.parent / "Market_Data" / "v2",
        MYDRIVE / "Market_Data" / "v2",
        MYDRIVE / "Kalman" / "Market_Data" / "v2",
        MYDRIVE / "kalman" / "Market_Data" / "v2",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    market_data = bounded_find_dir(
        MYDRIVE,
        lambda p: p.name == "Market_Data" and (p / "v2").exists(),
        max_depth=5,
    )
    return market_data / "v2" if market_data else None


def find_feature_root(model_root):
    candidates = [
        model_root.parent / "Market_Features" / "v2",
        MYDRIVE / "Market_Features" / "v2",
        MYDRIVE / "Kalman" / "Market_Features" / "v2",
        MYDRIVE / "kalman" / "Market_Features" / "v2",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    market_features = bounded_find_dir(
        MYDRIVE,
        lambda p: p.name == "Market_Features" and (p / "v2").exists(),
        max_depth=5,
    )
    return market_features / "v2" if market_features else None


def matrix_artifacts_complete(matrix_dir):
    required = []
    for market in ("us", "kr", "btc"):
        required.extend(
            [
                matrix_dir / f"{market}_matrix.parquet",
                matrix_dir / f"{market}_matrix_manifest.json",
            ]
        )
    return all(path.exists() for path in required)


def build_matrices(*, kpy, app_root, market_root, feature_root, matrix_dir, universe, spec):
    if market_root is None or not Path(market_root).exists():
        raise FileNotFoundError("Market_Data/v2 not found")
    if feature_root is None or not Path(feature_root).exists():
        raise FileNotFoundError("Market_Features/v2 not found")

    matrix_dir.mkdir(parents=True, exist_ok=True)
    run(
        [
            str(kpy),
            "-m",
            "research.model_v2.build_feature_matrix",
            "--market-root",
            str(market_root),
            "--feature-root",
            str(feature_root),
            "--universe",
            str(universe),
            "--spec",
            str(spec),
            "--output-dir",
            str(matrix_dir),
        ],
        cwd=app_root,
    )


def remap_server_path(raw_path, *, model_root, market_root, feature_root):
    raw = str(raw_path)
    p = Path(raw)
    if p.exists():
        return raw

    candidates = []

    if raw.startswith("/mnt/gdrive/"):
        suffix = raw[len("/mnt/gdrive/") :]
        candidates.extend(
            [
                MYDRIVE / suffix,
                model_root.parent / suffix,
            ]
        )

    if "Market_Data/" in raw and market_root is not None:
        suffix = raw.split("Market_Data/", 1)[1]
        candidates.append(Path(market_root).parent / suffix)

    if "Market_Features/" in raw and feature_root is not None:
        suffix = raw.split("Market_Features/", 1)[1]
        candidates.append(Path(feature_root).parent / suffix)

    if "Market_Model_V2/" in raw:
        suffix = raw.split("Market_Model_V2/", 1)[1]
        candidates.append(model_root / suffix)

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    return raw


def prepare_matrix_mirror(*, matrix_dir, model_root, market_root, feature_root, spec_payload):
    if MIRROR_DIR.exists():
        shutil.rmtree(MIRROR_DIR)
    MIRROR_DIR.mkdir(parents=True)

    unresolved = []

    for market in ("us", "kr", "btc"):
        parquet = matrix_dir / f"{market}_matrix.parquet"
        manifest = matrix_dir / f"{market}_matrix_manifest.json"

        if not parquet.exists() or not manifest.exists():
            unresolved.append(f"{market}:missing_matrix_or_manifest")
            continue

        shutil.copy2(parquet, MIRROR_DIR / parquet.name)

        payload = json.loads(manifest.read_text(encoding="utf-8"))
        remapped = {}
        for old_path, meta in payload.get("input_files", {}).items():
            new_path = remap_server_path(
                old_path,
                model_root=model_root,
                market_root=market_root,
                feature_root=feature_root,
            )
            remapped[new_path] = meta
        payload["input_files"] = remapped

        anchor_key = spec_payload["markets"][market.upper()]["anchor_key"]
        anchor_candidates = [
            path
            for path, meta in remapped.items()
            if str(meta.get("kind")) == "raw"
            and str(meta.get("fetch_key")) == anchor_key
        ]
        if not anchor_candidates or not any(
            Path(path).exists() for path in anchor_candidates
        ):
            unresolved.append(
                f"{market}:anchor={anchor_key}:candidates={anchor_candidates}"
            )

        (MIRROR_DIR / manifest.name).write_text(
            json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
            encoding="utf-8",
        )

    return unresolved


def coverage_report(kpy):
    script = r"""
import json
import sys
from pathlib import Path
import pandas as pd

root = Path(sys.argv[1])
out = {}

for market in ("us", "kr", "btc"):
    path = root / f"{market}_matrix.parquet"
    frame = pd.read_parquet(path, columns=["as_of", "target_label"])
    ts = pd.to_datetime(frame["as_of"], utc=True, errors="coerce")
    labeled = frame["target_label"].notna() & ts.notna()
    labeled_ts = ts.loc[labeled]

    out[market.upper()] = {
        "rows": int(len(frame)),
        "labeled_rows": int(labeled.sum()),
        "min_as_of": None if ts.dropna().empty else ts.dropna().min().isoformat(),
        "max_as_of": None if ts.dropna().empty else ts.dropna().max().isoformat(),
        "min_labeled_as_of": None if labeled_ts.empty else labeled_ts.min().isoformat(),
        "max_labeled_as_of": None if labeled_ts.empty else labeled_ts.max().isoformat(),
        "columns": list(frame.columns),
    }

print(json.dumps(out, ensure_ascii=False))
"""
    raw = capture([str(kpy), "-c", script, str(MIRROR_DIR)])
    return json.loads(raw.splitlines()[-1])


def preflight_coverage(coverage, spec_payload):
    errors = []
    start_year = int(START_DATE[:4])

    for market in ("US", "KR", "BTC"):
        info = coverage[market]
        horizon = int(spec_payload["markets"][market]["horizon_observations"])
        minimum_labeled = TRAIN_OBS + VALID_OBS + TEST_OBS + (2 * horizon)

        print(
            f"{market:4s} rows={info['rows']:,} "
            f"labeled={info['labeled_rows']:,} "
            f"min_labeled={info['min_labeled_as_of']} "
            f"max_labeled={info['max_labeled_as_of']} "
            f"required_labeled>={minimum_labeled}"
        )

        if info["labeled_rows"] < minimum_labeled:
            errors.append(
                f"{market}: labeled_rows={info['labeled_rows']} < {minimum_labeled}"
            )

        min_labeled = info["min_labeled_as_of"]
        if min_labeled is None:
            errors.append(f"{market}: no labeled history")
        else:
            min_year = int(min_labeled[:4])
            if min_year > start_year:
                errors.append(
                    f"{market}: earliest labeled year={min_year}, "
                    f"requested start year={start_year}"
                )

    return errors


def run_final_summary(kpy, output_dir):
    script = r"""
import json
import sys
from pathlib import Path
import pandas as pd

root = Path(sys.argv[1])

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

rows = []

for market in ("us", "kr", "btc"):
    perf = load_json(root / market / "historical_performance.json")
    if perf:
        rows.append({
            "method": market.upper(),
            "source": "MARKET_BACKTEST",
            "total_return": perf.get("total_return"),
            "cagr": perf.get("cagr"),
            "sharpe": perf.get("sharpe"),
            "max_drawdown": perf.get("max_drawdown"),
            "trade_count": perf.get("trade_count"),
        })

hrp = load_json(root / "portfolio" / "portfolio_performance.json")
if hrp:
    rows.append({
        "method": "HRP",
        "source": "PYPFOPT",
        "total_return": hrp.get("total_return"),
        "cagr": hrp.get("cagr"),
        "sharpe": hrp.get("sharpe"),
        "max_drawdown": hrp.get("max_drawdown"),
        "trade_count": None,
    })

risk_cmp = root / "riskfolio" / "riskfolio_comparison.csv"
if risk_cmp.exists():
    rdf = pd.read_csv(risk_cmp)
    for _, r in rdf.iterrows():
        if str(r.get("source", "")).upper() == "PYPFOPT":
            continue
        rows.append({
            "method": str(r.get("method")),
            "source": str(r.get("source", "RISKFOLIO")),
            "total_return": r.get("total_return"),
            "cagr": r.get("cagr"),
            "sharpe": r.get("sharpe"),
            "max_drawdown": r.get("max_drawdown"),
            "trade_count": None,
        })

df = pd.DataFrame(rows)
if not df.empty:
    raw = df.copy()
    printable = df.copy()
    for col in ("total_return", "cagr", "max_drawdown"):
        printable[col] = pd.to_numeric(printable[col], errors="coerce").map(
            lambda x: f"{x*100:.2f}%" if pd.notna(x) else "-"
        )
    printable["sharpe"] = pd.to_numeric(
        printable["sharpe"], errors="coerce"
    ).round(3)

    print("\nKALMAN HISTORICAL QUANT FINAL SUMMARY")
    print(printable.to_string(index=False))

    portfolio = raw.loc[raw["source"].isin(["PYPFOPT", "RISKFOLIO"])].copy()
    portfolio["sharpe"] = pd.to_numeric(portfolio["sharpe"], errors="coerce")
    portfolio = portfolio.dropna(subset=["sharpe"])
    if not portfolio.empty:
        best = portfolio.loc[portfolio["sharpe"].idxmax()]
        print(
            f"\nBEST PORTFOLIO SHARPE: "
            f"{best['method']} / {best['source']} / {best['sharpe']:.3f}"
        )

    raw.to_csv(root / "kalman_final_comparison.csv", index=False)

execution = load_json(root / "execution" / "execution_status.json")
validation = load_json(root / "historical_validation_report.json")

print("\nLEAN-INSPIRED SHADOW EXECUTION")
print(json.dumps(execution, ensure_ascii=False, indent=2, default=str))
print("\nVALIDATION")
print(json.dumps(validation, ensure_ascii=False, indent=2, default=str))
"""
    output = capture([str(kpy), "-c", script, str(output_dir)])
    print(output)


def main():
    phase("MOUNT DRIVE")
    drive.mount("/content/drive", force_remount=False)

    phase("FRESH REPO")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/kimtk94/Codex.git",
            str(REPO_ROOT),
        ]
    )

    app_root = REPO_ROOT / "kalman-toss-gateway"
    if not app_root.exists():
        raise FileNotFoundError(f"APP_ROOT missing: {app_root}")

    head = capture(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"]
    )
    print("Repo HEAD:", head)

    phase("ISOLATED KALMAN VENV")
    create_venv(KALMAN_VENV)
    kpy = KALMAN_VENV / "bin/python"
    kpip = KALMAN_VENV / "bin/pip"

    run(
        [
            str(kpip),
            "install",
            "-q",
            "pandas",
            "numpy",
            "scikit-learn",
            "pyarrow",
            "python-dotenv",
            "-r",
            str(app_root / "research/quant_stack/requirements-pypfopt.txt"),
        ]
    )

    if ENABLE_QLIB_RECORDER:
        run(
            [
                str(kpip),
                "install",
                "-q",
                "-r",
                str(app_root / "research/quant_stack/requirements-qlib.txt"),
            ]
        )

    env_check = capture(
        [
            str(kpy),
            "-c",
            (
                "import numpy,pandas,scipy,sklearn,pyarrow,pypfopt;"
                "print('OK', numpy.__version__, pandas.__version__, "
                "scipy.__version__, sklearn.__version__)"
            ),
        ]
    )
    print("KALMAN ENV:", env_check)

    phase("LOCATE DRIVE DATA")
    model_root = find_model_root()
    if model_root is None:
        raise FileNotFoundError(
            "Market_Model_V2 not found in Google Drive within search depth 5"
        )

    market_root = find_market_root(model_root)
    feature_root = find_feature_root(model_root)
    matrix_dir = model_root / "matrices"
    run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = model_root / "historical_quant_v3_colab" / run_tag

    spec = app_root / "config/model-v2-spec.json"
    universe = app_root / "config/market-data-v2-universe.json"
    spec_payload = json.loads(spec.read_text(encoding="utf-8"))

    print("MODEL_ROOT  :", model_root)
    print("MARKET_ROOT :", market_root)
    print("FEATURE_ROOT:", feature_root)
    print("MATRIX_DIR  :", matrix_dir)
    print("OUTPUT_DIR  :", output_dir)

    phase("MATRIX PREPARATION")
    rebuilt = False
    if FORCE_REBUILD_MATRICES or not matrix_artifacts_complete(matrix_dir):
        build_matrices(
            kpy=kpy,
            app_root=app_root,
            market_root=market_root,
            feature_root=feature_root,
            matrix_dir=matrix_dir,
            universe=universe,
            spec=spec,
        )
        rebuilt = True

    unresolved = prepare_matrix_mirror(
        matrix_dir=matrix_dir,
        model_root=model_root,
        market_root=market_root,
        feature_root=feature_root,
        spec_payload=spec_payload,
    )

    if unresolved and not rebuilt:
        print("Unresolved anchor manifests:", unresolved)
        print("Attempting one clean matrix rebuild...")
        build_matrices(
            kpy=kpy,
            app_root=app_root,
            market_root=market_root,
            feature_root=feature_root,
            matrix_dir=matrix_dir,
            universe=universe,
            spec=spec,
        )
        rebuilt = True
        unresolved = prepare_matrix_mirror(
            matrix_dir=matrix_dir,
            model_root=model_root,
            market_root=market_root,
            feature_root=feature_root,
            spec_payload=spec_payload,
        )

    if unresolved:
        raise RuntimeError(
            "PRECHECK_ANCHOR_OHLC_UNRESOLVED: " + " | ".join(unresolved)
        )

    phase("2017 COVERAGE PRECHECK")
    coverage = coverage_report(kpy)
    coverage_errors = preflight_coverage(coverage, spec_payload)

    if coverage_errors and not rebuilt:
        print("\nCoverage precheck failed on existing matrices.")
        print("Attempting one clean rebuild before stopping...")
        build_matrices(
            kpy=kpy,
            app_root=app_root,
            market_root=market_root,
            feature_root=feature_root,
            matrix_dir=matrix_dir,
            universe=universe,
            spec=spec,
        )
        rebuilt = True
        unresolved = prepare_matrix_mirror(
            matrix_dir=matrix_dir,
            model_root=model_root,
            market_root=market_root,
            feature_root=feature_root,
            spec_payload=spec_payload,
        )
        if unresolved:
            raise RuntimeError(
                "PRECHECK_ANCHOR_OHLC_UNRESOLVED_AFTER_REBUILD: "
                + " | ".join(unresolved)
            )
        coverage = coverage_report(kpy)
        coverage_errors = preflight_coverage(coverage, spec_payload)

    if coverage_errors:
        print("\n2017 historical precheck did not pass:")
        for item in coverage_errors:
            print(" -", item)
        raise RuntimeError(
            "PRECHECK_2017_HISTORY_INSUFFICIENT: historical feature backfill "
            "is required before a genuine 2017 walk-forward backtest."
        )

    phase("HISTORICAL WALK-FORWARD + HRP")
    output_dir.mkdir(parents=True, exist_ok=False)

    args = [
        str(kpy),
        "-m",
        "research.quant_stack.experiment_runner",
        "--matrix-dir",
        str(MIRROR_DIR),
        "--spec",
        str(spec),
        "--output-dir",
        str(output_dir),
        "--start-date",
        START_DATE,
        "--train",
        str(TRAIN_OBS),
        "--valid",
        str(VALID_OBS),
        "--test",
        str(TEST_OBS),
        "--max-hold-bars",
        str(MAX_HOLD_BARS),
        "--git-sha",
        head,
        "--portfolio-targets",
        "--portfolio-method",
        PORTFOLIO_METHOD,
        "--portfolio-lookback-days",
        str(PORTFOLIO_LOOKBACK),
        "--portfolio-min-observations",
        str(PORTFOLIO_MIN_OBS),
        "--portfolio-rebalance",
        PORTFOLIO_REBALANCE,
    ]

    if ENABLE_QLIB_RECORDER:
        args.extend(
            [
                "--qlib-recorder",
                "--qlib-tracking-root",
                str(model_root / "qlib_mlruns_colab_v3"),
                "--qlib-provider-root",
                "/content/qlib_provider_v3",
                "--qlib-experiment-name",
                "kalman_historical_quant_colab_v3",
            ]
        )

    run(args, cwd=app_root)

    phase("ARTIFACT VALIDATION")
    run(
        [
            str(kpy),
            "-m",
            "research.quant_stack.validate_artifacts",
            "--output-dir",
            str(output_dir),
        ],
        cwd=app_root,
    )

    phase("LEAN-INSPIRED SHADOW EXECUTION")
    run(
        [
            str(kpy),
            "-m",
            "research.quant_stack.lean_execution_runner",
            "--output-dir",
            str(output_dir),
            "--max-symbol-weight",
            "0.75",
            "--max-gross-weight",
            "1.0",
            "--min-order-notional",
            "10",
            "--max-single-order-fraction",
            "0.80",
            "--max-total-turnover-fraction",
            "2.0",
        ],
        cwd=app_root,
    )

    if RUN_RISKFOLIO:
        phase("ISOLATED RISKFOLIO BENCHMARK")
        create_venv(RISK_VENV)
        risk_py = RISK_VENV / "bin/python"
        risk_pip = RISK_VENV / "bin/pip"

        run(
            [
                str(risk_pip),
                "install",
                "-q",
                "-r",
                str(app_root / "research/quant_stack/requirements-riskfolio.txt"),
            ]
        )

        risk_env = os.environ.copy()
        risk_env["PYTHONPATH"] = str(app_root)

        run(
            [
                str(risk_py),
                "-m",
                "research.quant_stack.riskfolio_benchmark_runner",
                "--output-dir",
                str(output_dir),
                "--lookback-days",
                str(PORTFOLIO_LOOKBACK),
                "--min-observations",
                str(PORTFOLIO_MIN_OBS),
                "--rebalance",
                PORTFOLIO_REBALANCE,
            ],
            cwd=app_root,
            env=risk_env,
        )

    phase("FINAL SUMMARY")
    run_final_summary(kpy, output_dir)

    print("\n" + "=" * 88)
    print("COMPLETE")
    print("=" * 88)
    print("OUTPUT :", output_dir)
    print("RUN TAG:", run_tag)
    print("SAFETY : LIVE=False / Toss=False / Neon write=False")


try:
    main()
except Exception as exc:
    print("\n" + "!" * 88)
    print("KALMAN COLAB FAILED")
    print("!" * 88)
    print("FAILED PHASE:", CURRENT_PHASE)
    print("ERROR TYPE  :", type(exc).__name__)
    print("ERROR       :", str(exc))
    print("\nTRACEBACK")
    traceback.print_exc()
    raise
